# Gold / Silver vs Geopolitical Risk (1985-2025)

**Cel:** przygotować i sfine-tunować model przewidujący cenę złota/srebra na podstawie sytuacji geopolitycznej (wskaźnik GPR).

**Kroki:**
0. Instalacja zależności + pobranie danych z Kaggle.
1. Załadowanie i czyszczenie danych.
2. EDA — korelacja cen z GPR, szoki geopolityczne.
3. Inżynieria cech (lagi, z-score GPR, momentum).
4. Fine-tuning XGBoost (Optuna + TimeSeriesSplit).
5. Fine-tuning LSTM (PyTorch, wczesne zatrzymanie).
6. Analiza scenariuszy geopolitycznych.

## 0. Instalacja + pobranie danych

### 0.1 Instalacja zależności
Pierwsze uruchomienie — zainstaluj paczki z `requirements.txt`.
Uruchom tę komórkę raz; potem można ją pominąć.

In [ ]:
# Instalacja zależności — doinstaluje tylko to, czego brakuje / jest nieaktualne.
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

from packaging.requirements import Requirement

REQ_FILE = "requirements.txt"
missing = []
with open(REQ_FILE, encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        req = Requirement(line)
        try:
            inst = version(req.name)
            if req.specifier and inst not in req.specifier:
                missing.append(f"{line}  (zainstalowane: {inst})")
        except PackageNotFoundError:
            missing.append(line)

if missing:
    print("Brakujące / nieaktualne paczki:")
    for m in missing:
        print("  -", m)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", REQ_FILE])
    print("\n✓ Instalacja zakończona — zrestartuj kernel, jeśli importy się nie powiodą.")
else:
    print("✓ Wszystkie zależności są już zainstalowane — pomijam pip install.")

### 0.2 Pobranie danych z Kaggle

Używamy **`kagglehub`** — nowoczesny klient Kaggle, który obsługuje uwierzytelnianie
automatycznie (czyta `~/.kaggle/kaggle.json` albo zmienne `KAGGLE_USERNAME` / `KAGGLE_KEY`).

**Jeśli nie masz jeszcze tokenu:**
1. Wejdź na https://www.kaggle.com/settings → *Create New API Token* — pobierze się `kaggle.json`.
2. Umieść plik w:
   - Windows: `C:\Users\<user>\.kaggle\kaggle.json`
   - Linux/macOS: `~/.kaggle/kaggle.json`

**Alternatywa (bez pliku):** odkomentuj dwie linie z `os.environ` poniżej i wklej swoje dane.

In [ ]:
import os
import shutil
from pathlib import Path

# --- Opcja bez pliku kaggle.json: ---
# os.environ['KAGGLE_USERNAME'] = 'twoj_login_kaggle'
# os.environ['KAGGLE_KEY']      = 'twoj_api_key'
import kagglehub

DATASET = "shreyanshdangi/gold-silver-price-vs-geopolitical-risk-19852025"

# pobiera dataset do lokalnego cache kagglehub i zwraca ścieżkę
cache_path = kagglehub.dataset_download(DATASET)
print("Cache kagglehub:", cache_path)
print("Zawartość:", os.listdir(cache_path))

In [ ]:
# Kopiujemy CSV do lokalnego `data/` i aktualizujemy config
import yaml

PROJECT_ROOT = Path("").resolve()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

csv_files = list(Path(cache_path).glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Brak pliku CSV w {cache_path}. Zawartość: {os.listdir(cache_path)}")

src_csv = csv_files[0]
dst_csv = DATA_DIR / src_csv.name
shutil.copy2(src_csv, dst_csv)
print(f"Skopiowano: {src_csv.name}  ->  {dst_csv}")

# aktualizacja configs/config.yaml → paths.raw_csv
cfg_path = PROJECT_ROOT / "configs" / "config.yaml"
with open(cfg_path, encoding="utf-8") as fh:
    cfg_yaml = yaml.safe_load(fh)
cfg_yaml["paths"]["raw_csv"] = f"data/{src_csv.name}"
with open(cfg_path, "w", encoding="utf-8") as fh:
    yaml.safe_dump(cfg_yaml, fh, sort_keys=False, allow_unicode=True)
print("Zaktualizowano configs/config.yaml → paths.raw_csv =", cfg_yaml["paths"]["raw_csv"])

## 1. Załadowanie danych

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # moduły src.* używają ścieżek relatywnych do korzenia projektu

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

from src.data import load_config, load_processed, summary
from src.features import build_features

cfg = load_config(cfg_path)
df = load_processed(cfg, force=True)
print("Zakres:", df["date"].min(), "->", df["date"].max(), "| obs:", len(df))
df.head()

In [ ]:
summary(df)

## 2. EDA — ceny vs GPR

Sprawdzamy, czy skoki GPR (wojny, kryzysy) pokrywają się z ruchami cen złota i srebra.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(df["date"], df["gold"], label="Gold", color="goldenrod")
ax[0].plot(df["date"], df["silver"] * 80, label="Silver x80", color="silver", alpha=0.8)
ax[0].legend()
ax[0].set_title("Ceny metali szlachetnych")
ax[1].plot(df["date"], df["gpr"], color="crimson")
ax[1].set_title("Geopolitical Risk Index (GPR)")
plt.tight_layout()
plt.show()

In [ ]:
# Korelacja rocznych zmian
ann = df.set_index("date").resample("YE").last().pct_change().dropna()
corr = ann[["gold", "silver", "gpr"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Korelacja rocznych zmian")
plt.show()
corr

In [ ]:
# Jak metale zachowują się w okresach wysokiego GPR?
q = df["gpr"].quantile([0.25, 0.75]).to_dict()
df["gpr_regime"] = pd.cut(
    df["gpr"], bins=[-np.inf, q[0.25], q[0.75], np.inf], labels=["low", "mid", "high"]
)
df["gold_ret20"] = df["gold"].pct_change(20)
df["silver_ret20"] = df["silver"].pct_change(20)

df.groupby("gpr_regime", observed=True)[["gold_ret20", "silver_ret20"]].mean().mul(100).round(2)

## 3. Inżynieria cech

In [ ]:
feats = build_features(
    df.drop(columns=["gpr_regime", "gold_ret20", "silver_ret20"], errors="ignore"), cfg
)
print("Kształt:", feats.shape)
print("Target:", cfg["target"])
feats.tail()

## 4. Fine-tuning XGBoost (Optuna)

In [ ]:
from src.train import run_xgb

metrics_xgb = run_xgb(cfg)
metrics_xgb

## 5. Fine-tuning LSTM (PyTorch)

In [ ]:
from src.train import run_lstm

metrics_lstm = run_lstm(cfg)
metrics_lstm

## 6. Analiza scenariuszy

Co powie model, gdy w ostatnich 60 dniach GPR byłby 0.5×, 1×, 2×, 3× większy?

In [ ]:
from src.predict import feature_importances, predict_latest, scenario_predictions

print("Prognoza na najnowszych danych:")
print(predict_latest(cfg))

sc = scenario_predictions(cfg)
sc

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(sc["scenariusz"], sc["change_pct"])
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Prognozowana zmiana ceny [%]")
ax.set_title(f"Scenariusze GPR — horyzont {cfg['target']['horizon_days']} dni")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fi = feature_importances(cfg, top=20)
plt.figure(figsize=(7, 6))
sns.barplot(data=fi, y="feature", x="importance", palette="rocket")
plt.title("Top 20 cech (XGBoost)")
plt.tight_layout()
plt.show()
fi

## Wnioski

- XGBoost fine-tune (Optuna, TimeSeriesSplit) daje stabilną baseline na zwrotach logarytmicznych.
- LSTM uczy się zależności sekwencyjnych — warto porównać MAE i hit-rate.
- Analiza scenariuszy pokazuje wrażliwość modelu na zmianę reżimu GPR — kluczowe dla strategii *safe-haven*.
- Dalej: dodanie DXY, USD yields, VIX; uczenie wieloetapowe (multi-horizon).